In [78]:
import json
import requests
import pandas as pd
import numpy as np
import seaborn as sns
from boardlib_preprocessing import df_sql, parse_frames_to_holdset, convert_dataframe_to_holds, add_useability_features, upload_holds, flush_backup_holds, upload_climbs_batch

In [ ]:
query='SELECT * FROM holes'
df_holes = df_sql("decoy", query, index_col='id').astype({'product_id':int, 'name':str,'x':int,'y':int,'mirrored_hole_id':int,'mirror_group':int})
df_holes.head()

In [ ]:
sns.scatterplot(df_holes, x='x',y='y')

### Product 4 is the TB1, Product 5 is the TB2. We need to convert both of these hold-sets to real, physical holds. First step would be aligning them with their real, physical location on the board. Second step would be finding some way to manually add pull_dir+useability features.

I learned from the TB2 installation guide that these hole location measurements are in inches. Additionally, the TB2 DOESN'T HAVE A KICKBOARD! That's crazy, and a bit of a Mandela effect for me. I could swear it had one. But nope.

**The guide also comes with crystal clear hold orientations, and difficulty levels. This is exactly the data I'm looking for! Fuck yeah, Tension!** 

In [ ]:
query = "SELECT * FROM placements WHERE layout_id = 2"
df_placements = df_sql("decoy",query,index_col=None).astype({'id':int,'layout_id':int,'hole_id':int,'set_id':int,'default_placement_role_id':int})
df_placements = pd.merge(
    df_placements,
    df_holes,
    left_on='hole_id',
    right_on=df_holes.index,
    how='inner'
)
min_x = df_placements['x'].min()
df_placements.loc[df_placements['id'].isin([950, 935]), 'y'] += 8
df_placements['x_ft'] = (df_placements['x']+8.5-min_x)/12
df_placements['y_ft'] = (df_placements['y']+8)/12
df_placements['is_foot'] = (df_placements['default_placement_role_id']%4==0).astype(int)
df_placements = df_placements[['id','hole_id','x_ft','y_ft','is_foot']].sort_values('id')
print(
    df_placements['x_ft'].max(),
    df_placements['y_ft'].max()
)
df_placements.head()

In [ ]:
decoy_holds = convert_dataframe_to_holds(df_placements)
upload_holds('wall-9e000814e880',decoy_holds)

In [ ]:

flush_backup_holds("wall-e77a688049fb")

In [ ]:
query="""
        SELECT 
            c.uuid,
            c.name,
            cs.angle,
            c.frames,
            cs.difficulty_average,
            cs.quality_average,
            cs.ascensionist_count,
            cs.fa_username,
            c.created_at
        FROM climbs c
        JOIN climb_stats cs ON c.uuid = cs.climb_uuid
        WHERE c.layout_id = 2
            AND cs.ascensionist_count > 1
            AND cs.difficulty_average IS NOT NULL
            AND cs.quality_average IS NOT NULL
        ORDER BY cs.ascensionist_count DESC;
    """
df_decoy = df_sql("decoy",query)
upload_climbs_batch(
    df_decoy,
    'wall-9e000814e880'
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
grades_dict = {}
for i, row in df_climbs.iterrows():
    try:
        holdset = parse_frames_to_holdset(row['frames'])
        for h in [h for l in holdset.values() for h in l]:
            if h not in grades_dict:
                grades_dict[h] = []
            grades_dict[h].append(row['difficulty_average'])
    except:
        continue
scaler = MinMaxScaler(feature_range=(35,100))
df_hold_diff = pd.DataFrame(np.array([[k,1/np.mean(v)] for k, v in grades_dict.items()]), columns=['hold_index','difficulty_level']).set_index('hold_index')
df_hold_diff['difficulty_level'] = np.round(scaler.fit_transform(df_hold_diff[['difficulty_level']])/100,2)
df_hold_diff.sort_values('difficulty_level')